# Step 04 — three pseudo-sites: from here on, no data leaves a site

**Data type: RNA_array** (GSE65391). **Reads:** `step03_expression.rds`, `cohorts/site_assignment.csv`.
**Writes:** `step04_site_A.rds`, `step04_site_B.rds`, `step04_site_C.rds`, `step04_genes.rds`.

At the hackathon each institution keeps its own data. We imitate that by assigning the children in
GSE65391 at random to three sites, A, B and C. **This is the last notebook that holds every sample in
one matrix.** From here on:

- each site's data sits in its own file, and code reads one site's file at a time;
- a site sends only summaries: per-gene counts and sums, model coefficients, counts per cluster;
- a **coordinator** combines what the sites send and returns shared parameters.

Every message a site sends passes through `send()` in `src/federation.R`, which records its size and
shape. Step 17 audits the whole record.

**Simulation-only oracle.** Because the sites are simulated, we can also compute some quantities on
all samples together. Such cells are headed "Oracle" and serve only as proof or yardstick. Nothing an
oracle computes feeds a later step.

In [1]:
source("../src/paths.R")
source("../src/federation.R")
start_log("04")
s3 <- readRDS(art("step03_expression.rds"))
people <- unique(s3$meta[, c("subject", "disease")])
nrow(people)

[1] 204

## Assign children, not samples

All visits of one child go to one site, as in real life. The draw is stratified by disease so that
every site has its own healthy controls. **The assignment is frozen:** written once to
`cohorts/site_assignment.csv`, which is committed, and read back on every later run.

In [2]:
SITE_FILE <- coh("site_assignment.csv")
if (file.exists(SITE_FILE)) {
  a <- read.csv(SITE_FILE, stringsAsFactors = FALSE)
  stopifnot(setequal(a$subject, people$subject))
  cat("site assignment read from", SITE_FILE, "\n")
} else {
  set.seed(SEED)
  people$site <- NA_character_
  for (g in levels(people$disease)) {
    i <- which(people$disease == g)
    people$site[sample(i)] <- rep_len(SITES, length(i))
  }
  a <- people[order(people$subject), c("subject", "disease", "site")]
  write.csv(a, SITE_FILE, row.names = FALSE)
  cat("site assignment drawn and frozen to", SITE_FILE, "\n")
}
site <- setNames(a$site, a$subject)[s3$meta$subject]
addmargins(table(disease = s3$meta$disease, site = site))

site assignment read from /Users/adeslatt/Scitechcon Dropbox/Anne DeslattesMays/projects/endotypes-transcriptomics/cohorts/site_assignment.csv 


,A,B,C,Sum
Healthy,16,16,16,48
SLE,322,264,338,924
Sum,338,280,354,972


## Hand each site its own data

In [3]:
for (s in SITES) {
  j <- which(site == s)
  saveRDS(list(site = s, E = s3$E[, j], meta = cbind(s3$meta[j, ], site = s)),
          site_file("04", s))
}
rm(s3); invisible(gc())                     # the pooled matrix is gone from here on
file.size(sapply(SITES, site_file, step = "04"))

[1] 38536020 31973477 40607068

## First federated exchange: which genes can every site use?

A gene stuck at the array floor in every sample of a site has no variance there, and no correction
can rescale it. Each site sends one yes/no flag per gene: "this gene varies here". The coordinator
keeps the genes that vary at every site, and each site drops the rest.

In [4]:
flags <- lapply(SITES, function(s) {
  d <- readRDS(site_file("04", s))
  send(apply(d$E, 1, var) > 0, s, "gene varies at site (flag)", ncol(d$E))
})
keep <- Reduce(`&`, flags)
c(genes = length(keep), dropped = sum(!keep), kept = sum(keep))

genes dropped    kept 
  28948     964   27984

In [5]:
s3g <- readRDS(art("step03_expression.rds"))[c("expressed", "genes")]
genes <- list(keep = names(keep)[keep], expressed = s3g$expressed[names(keep)[keep]])
stopifnot(all(read_gene_set("ifn-type1-6.txt") %in% genes$keep))
saveRDS(genes, art("step04_genes.rds"))
for (s in SITES) {
  d <- readRDS(site_file("04", s)); d$E <- d$E[genes$keep, ]
  saveRDS(d, site_file("04", s))
}
c(genes_kept = length(genes$keep), expressed_kept = sum(genes$expressed))

genes_kept expressed_kept 
         27984           8825

## Findings

Each site holds about a third of the children and 16 healthy controls. The only message so far is one
flag per gene from each site. Genes flat at any site were dropped everywhere; all six interferon-score
genes remain.